If your body is a system, protein kinases behave like tiny machines. There's a type of protein EGFR (Epidermal Growth Factor) that acts like a light switch that causes cells to grow; sometimes the switch gets stuck and leads to cancer.


When modeling the X input will be the EGFR features (phospho, protein, RNA, activity) but the mutation type will influence this.

In [63]:
#pip install cptac
import pandas as pd
import cptac

In [90]:
mutation = pd.read_csv('data/egfr_mutation.csv')
"""
removed WT; no EGFR mutation present
if needed it's mutations_raw.csv

EFGR = mutation
"""
mutation.head() #EGFR_type and TCGA patient ID (need to mask to match)

#CPTAC data
luad_data = cptac.Luad()
proteomics = luad_data.get_proteomics(source='bcm')
phospho = luad_data.get_phosphoproteomics(source='bcm')
rna_seq = luad_data.get_transcriptomics(source='bcm')
#somatic_mutations = luad_data.get_somatic_mutation()

egfr_rna = rna_seq[["EGFR"]]
egfr_proteomics = proteomics[["EGFR"]]

Why do we need all four types of CPTAC data? 

Phospho 
- capture activation state (site-specfic)
   - Y1068
   - Y1173
   - Y992

Protein
- total EGFR abundance 
- baseline feature

rNA
- expression level

Activity
- Functional output

In [95]:
"""
downloaded from checkpoint_3_datacol:
log2 ref normalized proteomic
log2 red tumor/normal proteomic
log2 ref normalized phosphoproteomic
log2 red tumor/normal phosphoproteomic
log2 ref normalized transcriptomic
log2 red tumor/normal transcriptomic

"""
#phosphoproteomics data with EGFR phospho sites of interest (Y1172, Y1173, Y1069, Y1068)
egfr_phospho_tcga = pd.read_csv("data/egfr_phospho_tcga.csv")
egfr_phospho_tcga = egfr_phospho_tcga.fillna(0)


mutation['PATIENT_ID'] = mutation['PATIENT_ID'].str.strip().str.upper()
egfr_phospho_tcga['PATIENT_ID'] = egfr_phospho_tcga['PATIENT_ID'].str.strip().str.upper()

#merged = pd.merge(mut_exploded, egfr_phospho_tcga, left_on='PATIENT_ID', right_on='PATIENT_ID', how='inner')
#merged.head()


In [111]:
#combining with TCGA_ID and CPTAC_ID (add to datacleaning)
mutation_ids = mutation['PATIENT_ID'].tolist()
mutation_ids= sorted(list(set(mutation['PATIENT_ID'].unique())))
phospho_ids = egfr_phospho_tcga['PATIENT_ID'].tolist()
phospho_ids = sorted(list(set(egfr_phospho_tcga['PATIENT_ID'].unique())))


common_ids = sorted(list(set(mutation_ids).intersection(set(phospho_ids))))
only_mutation_ids = [id for id in mutation_ids if id not in phospho_ids]
only_phospho_ids = [id for id in phospho_ids if id not in mutation_ids]
#print(f"Number of common patient IDs: {len(common_ids)}") #zero
print(f"Number of patient IDs only in mutation dataset: {len(only_mutation_ids)}") #zero
print(f"Number of patient IDs only in phosphoproteomics dataset: {len(only_phospho_ids)}") #zero


max_len= max(len(only_mutation_ids), len(only_phospho_ids), len(common_ids))

comparison_df = pd.DataFrame({
    'Common_IDs': pd.Series(common_ids),
    'Only_in_Mutation_CSV': pd.Series(list(only_mutation_ids)[:max_len]), # Showing first few if too many
    'Only_in_Phospho_CPTAC': pd.Series(list(only_phospho_ids)[:max_len])
})

#comparison_df.to_csv('data/patient_id_comparison.csv', index=False)
all_ids = pd.read_csv('data/patient_id_comparison.csv')
all_ids.head()

Number of patient IDs only in mutation dataset: 70
Number of patient IDs only in phosphoproteomics dataset: 207


,Common_IDs,Only_in_Mutation_CSV,Only_in_Phospho_CPTAC
0,NaN,TCGA-05-4382-01,C3L-00001
1,NaN,TCGA-05-4402-01,C3L-00001.N
2,NaN,TCGA-05-4410-01,C3L-00009
3,NaN,TCGA-05-5423-01,C3L-00009.N
4,NaN,TCGA-17-Z026-01,C3L-00080


In [114]:
#padding with NaN for missing values
df1 = mutation[['PATIENT_ID']].reset_index(drop=True)
df2 = egfr_phospho_tcga[['PATIENT_ID']].reset_index(drop=True)
side_by_side = pd.concat([df1, df2], axis=1) 
#side_by_side.to_csv('data/ids_side_by_side.csv', index=False)
all_ids = pd.read_csv('data/ids_side_by_side.csv')
all_ids.head()

,PATIENT_ID,PATIENT_ID.1
0,TCGA-05-4382-01,C3L-00001
1,TCGA-05-4402-01,C3L-00009
2,TCGA-05-4410-01,C3L-00080
3,TCGA-05-5423-01,C3L-00083
4,TCGA-17-Z026-01,C3L-00093
